In [1]:
import pandas as pd
import os
import pandas as pd
from pathlib import Path

In [2]:
kg = pd.read_csv("../knowledge_graph/shep_kg_from_to_rel.tsv", sep="\t")
kg['nt_from'] = kg["from"].apply(lambda x: x.rsplit("_", 1)[0])
kg['nt_to'] = kg["to"].apply(lambda x: x.rsplit("_", 1)[0])

In [3]:
counts = kg.groupby(['nt_from', 'nt_to', 'rel']).size().reset_index(name='count')
bipartites = counts[counts["nt_from"]!=counts['nt_to']]

In [4]:
negative_rels = {
    "disease_phenotype_negative",
    "contraindication",
    "off-label use",
    "protein_absent_anatomy"
}

bipartites = bipartites.copy()
bipartites["sign"] = bipartites['rel'].apply(lambda x: "negative" if x in negative_rels else "postive")


In [5]:
out_dir = Path("~/dr_benchmark/mxr_baseline/mxr_networks/bipartite_networks").expanduser()
out_dir.mkdir(parents=True, exist_ok=True)

rel_sign = bipartites.set_index("rel")["sign"].to_dict()
rel_sign = {k: +1 if v.strip().lower() == "postive" else -1 for k, v in rel_sign.items()}

pairs = bipartites[["nt_from", "nt_to"]].drop_duplicates()

for _, row in pairs.iterrows():
    nt_from = row["nt_from"]
    nt_to = row["nt_to"]

    rels = bipartites[
        (bipartites.nt_from == nt_from) & (bipartites.nt_to == nt_to)
    ]["rel"].tolist()

    sub = kg[kg["rel"].isin(rels)].copy()

    sub["weight"] = sub["rel"].map(rel_sign)

    sub_out = sub[["from", "to", "weight"]]

    name = f"{nt_from.replace(' ', '_').replace('-', '_').replace('/', '_')}__{nt_to.replace(' ', '_').replace('-', '_').replace('/', '_')}"
    out_path = out_dir / f"{name}.tsv"

    sub_out.to_csv(out_path, sep="\t", index=False, header=False)


In [6]:
Path("~/dr_benchmark/mxr_baseline/mxr_networks/multiplex_networks").expanduser().mkdir(parents=True, exist_ok=True)

bipartite_pairs = set(zip(bipartites.nt_from, bipartites.nt_to))
all_pairs = set(zip(counts.nt_from, counts.nt_to))
multiplex_pairs = all_pairs - bipartite_pairs

processed_pairs = set()

for nt_from, nt_to in multiplex_pairs:
    if (nt_to, nt_from) in processed_pairs:
        continue
    processed_pairs.add((nt_from, nt_to))

    rels = counts[(counts.nt_from == nt_from) & (counts.nt_to == nt_to)]['rel'].tolist()
    rels += counts[(counts.nt_from == nt_to) & (counts.nt_to == nt_from)]['rel'].tolist()

    name = f"{nt_from.replace('/', '_').replace(' ', '_').replace('-', '_')}__{nt_to.replace('/', '_').replace(' ', '_').replace('-', '_')}"
    dir_path = Path(f"~/dr_benchmark/mxr_baseline/mxr_networks/multiplex_networks/{name}")
    dir_path.expanduser().mkdir(parents=True, exist_ok=True)

    for rel in rels:
        sub = kg[kg["rel"] == rel].copy()
        sub["weight"] = 1
        sub_out = sub[["from", "to", "weight"]]
        rel = rel.replace(' ', '_').replace('-', '_')
        out_path = dir_path / f"{rel}.tsv"
        sub_out.to_csv(out_path, sep="\t", index=False, header=False)


In [7]:
orphan_drugs = pd.read_csv("../knowledge_graph/orphan_associations.tsv", sep="\t")

In [8]:
config_dir = Path("~/dr_benchmark/mxr_baseline/mxr_config").expanduser()
seeds_dir = Path("~/dr_benchmark/mxr_baseline/mxr_seeds").expanduser()
config_dir.mkdir(parents=True, exist_ok=True)
seeds_dir.mkdir(parents=True, exist_ok=True)

In [9]:
config_template = """
self_loops: 0
r: 0.7
eta: [0,0,0,0,1,0,0,0,0,0]
lamb:
    - [0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1]
    - [0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1]
    - [0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1]
    - [0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1]
    - [0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1]
    - [0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1]
    - [0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1]
    - [0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1]
    - [0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1]
    - [0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1]
multiplex:
    Anatomy:
        layers:
            - mxr_networks/multiplex_networks/anatomy__anatomy/anatomy_anatomy.tsv
        delta: 0
        graph_type: [01]
        tau: [1]
    BioProc:
        layers:
            - mxr_networks/multiplex_networks/biological_process__biological_process/bioprocess_bioprocess.tsv
        delta: 0
        graph_type: [01]
        tau: [1]
    CellComp:
        layers:
            - mxr_networks/multiplex_networks/cellular_component__cellular_component/cellcomp_cellcomp.tsv
        delta: 0
        graph_type: [01]
        tau: [1]
    Disease:
        layers:
            - mxr_networks/multiplex_networks/disease__disease/disease_disease.tsv
        delta: 0
        graph_type: [01]
        tau: [1]
    Drug:
        layers:
            - mxr_networks/multiplex_networks/drug__drug/drug_drug.tsv
        delta: 0
        graph_type: [01]
        tau: [1]        
    Phenotype:
        layers:
            - mxr_networks/multiplex_networks/effect_phenotype__effect_phenotype/phenotype_phenotype.tsv 
        delta: 0
        graph_type: [01]
        tau: [1]      
    Exposure:
        layers:
            - mxr_networks/multiplex_networks/exposure__exposure/exposure_exposure.tsv
        delta: 0
        graph_type: [01]
        tau: [1]    
    Protein:
        layers:
            - mxr_networks/multiplex_networks/gene_protein__gene_protein/protein_protein.tsv
        delta: 0
        graph_type: [01]
        tau: [1]    
    MolFunc:
        layers:
            - mxr_networks/multiplex_networks/molecular_function__molecular_function/molfunc_molfunc.tsv
        delta: 0
        graph_type: [01]
        tau: [1]    
    Pathway:
        layers:
            - mxr_networks/multiplex_networks/pathway__pathway/pathway_pathway.tsv
        delta: 0
        graph_type: [01]
        tau: [1]                          
bipartite:
    mxr_networks/bipartite_networks/gene_protein__pathway.tsv: {{'source': 'Protein', 'target': 'Pathway', graph_type: 01}}
    mxr_networks/bipartite_networks/gene_protein__molecular_function.tsv: {{'source': 'Protein', 'target': 'MolFunc', graph_type: 01}}	
    mxr_networks/bipartite_networks/gene_protein__cellular_component.tsv: {{'source': 'Protein', 'target': 'CellComp', graph_type: 01}}	
    mxr_networks/bipartite_networks/gene_protein__biological_process.tsv: {{'source': 'Protein', 'target': 'BioProc', graph_type: 01}}	
    mxr_networks/bipartite_networks/gene_protein__anatomy.tsv: {{'source': 'Protein', 'target': 'Anatomy', graph_type: 01}}	
    mxr_networks/bipartite_networks/exposure__molecular_function.tsv: {{'source': 'Exposure', 'target': 'MolFunc', graph_type: 01}}	                
    mxr_networks/bipartite_networks/exposure__gene_protein.tsv: {{'source': 'Exposure', 'target': 'Protein', graph_type: 01}}
    mxr_networks/bipartite_networks/exposure__disease.tsv: {{'source': 'Exposure', 'target': 'Disease', graph_type: 01}}
    mxr_networks/bipartite_networks/exposure__cellular_component.tsv: {{'source': 'Exposure', 'target': 'CellComp', graph_type: 01}} 
    mxr_networks/bipartite_networks/exposure__biological_process.tsv: {{'source': 'Exposure', 'target': 'BioProc', graph_type: 01}}        
    mxr_networks/bipartite_networks/effect_phenotype__gene_protein.tsv: {{'source': 'Phenotype', 'target': 'Protein', graph_type: 01}}            
    mxr_networks/bipartite_networks/drug__gene_protein.tsv: {{'source': 'Drug', 'target': 'Protein', graph_type: 01}}         
    mxr_networks/bipartite_networks/drug__effect_phenotype.tsv: {{'source': 'Drug', 'target': 'Phenotype', graph_type: 01}}    
    mxr_networks/bipartite_networks/drug__disease.tsv: {{'source': 'Drug', 'target': 'Disease', graph_type: 01}} 
    mxr_networks/bipartite_networks/disease__gene_protein.tsv: {{'source': 'Disease', 'target': 'Protein', graph_type: 01}}            	
    mxr_networks/bipartite_networks/disease__effect_phenotype.tsv: {{'source': 'Disease', 'target': 'Phenotype', graph_type: 01}}            
"""

In [10]:
unique_from_nodes = orphan_drugs["from"].unique()

for i, node_from in enumerate(unique_from_nodes, start=1):
    seed_path = seeds_dir / f"seed_{i}.txt"
    with open(seed_path, "w") as fseed:
        fseed.write(str(node_from) + "\n")
    
    # Créer le fichier config
    config_path = config_dir / f"config_{i}.yaml"
    with open(config_path, "w") as fconf:
        # On ajoute la ligne seed au début
        fconf.write(f"seed: mxr_seeds/seed_{i}.txt\n")
        fconf.write(config_template.lstrip())  # enlever les retours à la ligne superflus au début

print(f"Generated {len(unique_from_nodes)} seed and config files.")

Generated 411 seed and config files.
